In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

data_dir = r'archive'


"""
Reads Olist e-commerce datasets from a directory and merges them 
into a central master dataframe.
"""
print("Loading datasets...")

# Load all required datasets
translations = pd.read_csv(os.path.join(data_dir, 'product_category_name_translation.csv'))
orders = pd.read_csv(os.path.join(data_dir, 'olist_orders_dataset.csv'))
order_items = pd.read_csv(os.path.join(data_dir, 'olist_order_items_dataset.csv'))
products = pd.read_csv(os.path.join(data_dir, 'olist_products_dataset.csv'))
customers = pd.read_csv(os.path.join(data_dir, 'olist_customers_dataset.csv'))
geolocation = pd.read_csv(os.path.join(data_dir, 'olist_geolocation_dataset.csv'))
payments = pd.read_csv(os.path.join(data_dir, 'olist_order_payments_dataset.csv'))
reviews = pd.read_csv(os.path.join(data_dir, 'olist_order_reviews_dataset.csv'))
sellers = pd.read_csv(os.path.join(data_dir, 'olist_sellers_dataset.csv'))
print("Datasets loaded successfully.")

Loading datasets...
Datasets loaded successfully.


In [3]:
list_of_dfs = [orders, order_items, products, customers, geolocation, payments, reviews, sellers, translations]
name = ['orders', 'order_items', 'products', 'customers', 'geolocation', 'payments', 'reviews', 'sellers', 'translations']
for x in range(len(list_of_dfs)):
    df = list_of_dfs[x]
    df_name = name[x]
    print('name:', df_name)
    print(f"{df.shape[0]} rows, {df.shape[1]} columns")
    
    print("-" * 50)
    print(df.head(2))
    print("-" * 50)
    print(df.info())
    print("-" * 100)
    

name: orders
99441 rows, 8 columns
--------------------------------------------------
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31:00           2018-08-07 15:27:45   

  order_estimated_delivery_date  
0           2017-10-18 00:00:00  
1           2018-08-13 00:00:00  
--------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Co

In [5]:
# --- 2. Clean Geolocation Data ---
# The geolocation dataset has multiple coordinates for the same zip code.
# To avoid duplicating rows during the merge, we will take the average latitude/longitude per zip code.
geo_grouped = geolocation.groupby('geolocation_zip_code_prefix').agg({
    'geolocation_lat': 'mean',
    'geolocation_lng': 'mean'
}).reset_index()

# --- 3. Merge Datasets Step-by-Step ---
print("Merging datasets into Master Dataframe...")

# Step A: Connect Orders to Customers
master_df = pd.merge(orders, customers, on='customer_id', how='left')

# Step B: Connect to Order Items (This expands rows if an order has multiple items)
master_df = pd.merge(master_df, order_items, on='order_id', how='left')

# Step C: Connect to Products
master_df = pd.merge(master_df, products, on='product_id', how='left')

# Step D: Add English Product Category Names
master_df = pd.merge(master_df, translations, on='product_category_name', how='left')

# Step E: Connect to Sellers
master_df = pd.merge(master_df, sellers, on='seller_id', how='left')

# Step F: Connect to Order Payments 
# (This expands rows if an order was paid using multiple methods, e.g., Card + Voucher)
master_df = pd.merge(master_df, payments, on='order_id', how='left')

# Step G: Connect to Reviews
master_df = pd.merge(master_df, reviews, on='order_id', how='left')

# Step H: Connect Customer Geolocation
master_df = pd.merge(master_df, geo_grouped, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
master_df.rename(columns={'geolocation_lat': 'customer_lat', 'geolocation_lng': 'customer_lng'}, inplace=True)
master_df.drop('geolocation_zip_code_prefix', axis=1, inplace=True)

# Step I: Connect Seller Geolocation
master_df = pd.merge(master_df, geo_grouped, left_on='seller_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='left')
master_df.rename(columns={'geolocation_lat': 'seller_lat', 'geolocation_lng': 'seller_lng'}, inplace=True)
master_df.drop('geolocation_zip_code_prefix', axis=1, inplace=True)

# --- 4. Final Cleanup & Export ---
# Drop the original Portuguese category name since we have the English one
if 'product_category_name_english' in master_df.columns:
    master_df.drop('product_category_name', axis=1, inplace=True)
    master_df.rename(columns={'product_category_name_english': 'product_category'}, inplace=True)

print(f"Master Dataset Shape: {master_df.shape}")

# Save the unified dataset to your directory
output_path = os.path.join(data_dir, 'olist_master_dataset.csv')
master_df.to_csv(output_path, index=False)
print(f"Master dataset successfully saved to: {output_path}")

Merging datasets into Master Dataframe...
Master Dataset Shape: (119143, 43)
Master dataset successfully saved to: C:\Users\khaled\Desktop\New folder\archive\olist_master_dataset.csv


In [6]:
master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_lat,customer_lng,seller_lat,seller_lng
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,-23.576983,-46.587161,-23.680729,-46.444238
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,-23.576983,-46.587161,-23.680729,-46.444238
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,-23.576983,-46.587161,-23.680729,-46.444238
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,-12.177924,-44.660711,-19.807681,-43.980427
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,-16.745150,-48.514783,-21.363502,-48.229601


In [7]:
master_df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category', 'seller_zip_code_prefix', 'seller_city',
       'seller_state', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'review_id', 'review_score',
       'review_comment_title', 'review_comment_message',
       'review_creation_date', 'review_answer_timestamp', 'customer_lat',
       'customer_lng', 'seller_lat', 'seller_lng'],
      dtype='object')

In [8]:
# Instead of a direct merge, aggregate payments by order_id first
payments_agg = payments.groupby('order_id').agg({
    'payment_value': 'sum', # Total amount paid for the order
    'payment_installments': 'max', # Max installments used
    'payment_type': lambda x: ', '.join(x.unique()) # List of methods used (e.g., 'credit_card, voucher')
}).reset_index()

# Now merge this aggregated table into your master dataframe
master_df = pd.merge(master_df, payments_agg, on='order_id', how='left')

In [9]:
# 1. Convert string dates to actual Datetime objects
datetime_cols = [
    'order_purchase_timestamp', 'order_approved_at', 
    'order_delivered_carrier_date', 'order_delivered_customer_date', 
    'order_estimated_delivery_date', 'shipping_limit_date'
]
for col in datetime_cols:
    master_df[col] = pd.to_datetime(master_df[col])

# 2. Operations Insight: Calculate actual delivery time (in days)
master_df['delivery_time_days'] = (master_df['order_delivered_customer_date'] - master_df['order_purchase_timestamp']).dt.days

# 3. Operations Insight: Flag delayed orders (1 = Delayed, 0 = On Time)
master_df['is_delayed'] = (master_df['order_delivered_customer_date'] > master_df['order_estimated_delivery_date']).astype(int)

# 4. Sales Insight: Extract Year and Month for the Predictive Forecasting engine
master_df['purchase_year_month'] = master_df['order_purchase_timestamp'].dt.to_period('M')

In [10]:
def haversine_distance(lat1, lon1, lat2, lon2):
    r = 6371 # Radius of earth in kilometers
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    
    a = np.sin(delta_phi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    
    return r * c

# Apply to your master dataframe
master_df['logistics_distance_km'] = haversine_distance(
    master_df['seller_lat'], master_df['seller_lng'],
    master_df['customer_lat'], master_df['customer_lng']
)

In [11]:
# 1. Check if the column is missing, and if so, merge it in!
if 'payment_value' not in master_df.columns:
    print("Column 'payment_value' missing. Merging payments data now...")
    
    # Aggregate payments by order_id to prevent row duplication
    payments_agg = payments.groupby('order_id').agg({
        'payment_value': 'sum'
    }).reset_index()
    
    # Merge into master_df
    master_df = pd.merge(master_df, payments_agg, on='order_id', how='left')
    print("Merge complete!")

# 2. Handle any NaN values that might have appeared for orders without payment info
master_df['payment_value'] = master_df['payment_value'].fillna(0)

# 3. Now run your RFM Analysis safely
snapshot_date = master_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

rfm = master_df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days, # Recency
    'order_id': 'nunique', # Frequency
    'payment_value': 'sum' # Monetary
}).reset_index()

rfm.rename(columns={
    'order_purchase_timestamp': 'Recency',
    'order_id': 'Frequency',
    'payment_value': 'Monetary'
}, inplace=True)

print("RFM Analysis successful!")
display(rfm.head())

Column 'payment_value' missing. Merging payments data now...
Merge complete!
RFM Analysis successful!


,customer_unique_id,Recency,Frequency,Monetary
0,0000366f3b9a7992bf8c76cfdf3221e2,161,1,141.90
1,0000b849f77a49e4a4ce2b2a4ca5be3f,164,1,27.19
2,0000f46a3911fa3c0805444483337064,586,1,86.22
3,0000f6ccb0745a6a4b88665a16c9f078,370,1,43.62
4,0004aac84e0df4da2b147fca70cf8255,337,1,196.89


In [12]:
import pandas as pd
import os

# 1. Self-Healing Check: Merge 'payment_value' if Jupyter forgot it
if 'payment_value' not in master_df.columns:
    print("⚠️ 'payment_value' missing! Merging payments data now...")
    payments_agg = payments.groupby('order_id').agg({'payment_value': 'sum'}).reset_index()
    master_df = pd.merge(master_df, payments_agg, on='order_id', how='left')
    master_df['payment_value'] = master_df['payment_value'].fillna(0)
    print("✅ Payments merged successfully!")

# 2. Self-Healing Check: Ensure datetime format
master_df['order_purchase_timestamp'] = pd.to_datetime(master_df['order_purchase_timestamp'])

print("⏳ Building Daily Forecasting Dataset...")

# 3. Create the Daily Sales & Volume dataset
daily_forecasting_df = master_df.groupby(master_df['order_purchase_timestamp'].dt.date).agg(
    total_sales=('payment_value', 'sum'),
    total_orders=('order_id', 'nunique'),
    unique_customers=('customer_unique_id', 'nunique'),
    avg_freight=('freight_value', 'mean')
).reset_index()

# 4. Clean up the output
daily_forecasting_df.rename(columns={'order_purchase_timestamp': 'date'}, inplace=True)
daily_forecasting_df['date'] = pd.to_datetime(daily_forecasting_df['date'])

print("🎉 Forecasting dataset created successfully!")
display(daily_forecasting_df.head())

# Optional: Save this to your folder for the AI model to use later
# output_path = os.path.join(data_dir, 'daily_sales_forecast_data.csv')
# daily_forecasting_df.to_csv(output_path, index=False)
# print(f"Saved to {output_path}")

⏳ Building Daily Forecasting Dataset...
🎉 Forecasting dataset created successfully!


,date,total_sales,total_orders,unique_customers,avg_freight
0,2016-09-04,272.46,1,1,31.67
1,2016-09-05,75.06,1,1,15.56
2,2016-09-13,40.95,1,1,NaN
3,2016-09-15,0.00,1,1,2.83
4,2016-10-02,109.34,1,1,9.34
